In [4]:
import os

from langchain.chat_models import init_chat_model


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [5]:
model = init_chat_model("groq:qwen/qwen3-32b")

In [6]:
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026F43DE0150>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026F44EFFB10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="the title of the movie")
    year:str = Field(description="the year movie was released")
    genre:str = Field(description="genre of the movie")
    director:str = Field(description="the director of the movie")


In [8]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure.invoke("give me details of the movie wolf of thr wall street")

Movie(title='The Wolf of Wall Street', year='2013', genre='Biographical Drama', director='Martin Scorsese')

TypedDict


In [9]:
from typing import TypedDict,Annotated

In [37]:
class Actor(TypedDict):
    """an actor with details"""
    name:Annotated[str,...,"the name of the actor"]
    age:Annotated[int,...,"the age of the actor"]

In [38]:
class Movies(TypedDict):
    """a movies with details"""
    title:Annotated[str,...,"the title of the movie"]
    year:Annotated[str,...,"the year movie was released"]
    genre:Annotated[str,...,"genre of the movie"]
    director:Annotated[str,...,"the director of the movie"]
    cast:Annotated[list[Actor],...,"the cast of the movie"]

In [39]:
model_with_structures = model.with_structured_output(Movies)

In [40]:
response = model_with_structures.invoke("give me details of the movie wolf of thr wall street")

In [41]:
response

{'cast': [{'age': 39, 'name': 'Leonardo DiCaprio'},
  {'age': 32, 'name': 'Jonah Hill'},
  {'age': 26, 'name': 'Margot Robbie'}],
 'director': 'Martin Scorsese',
 'genre': 'Biographical Drama',
 'title': 'The Wolf of Wall Street',
 'year': '2013'}

In [42]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

In [55]:
from langchain.agents import create_agent
from typing_extensions import TypedDict,Annotated

In [56]:
class ContactInfo(TypedDict):
    name: Annotated[str, "the name of the person"]
    email: Annotated[str, "the email of the person"]
    phone: Annotated[str, "the phone number of the person"]

In [57]:
agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

In [47]:
response = agent.invoke({"messages":[{"role":"user","content":"extract the contact info from this text: 'John Doe, email: john.doe@example.com, phone: 123-456-7890'"}]})

In [58]:
response["structured_response"]

ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')